# Capstone 03 - Day 3 - Gold layer (dimensions, facts, KPI marts, reconciliation)

Run the cells **top to bottom, once**, after `02_silver_quality_rebuilt` has passed its final validation. It reads **only** the Silver tables (and the same airline / airport reference tables Silver used) and writes everything to `capstone_3flight.gold_schema`.

**Tables this notebook creates**

| Table | Grain (one row per) | Purpose |
|---|---|---|
| `dim_airline` | airline_id (+1 UNKNOWN member, sk = -1) | Conformed airline dimension, hash surrogate key `airline_sk` |
| `dim_airport` | airport_id (+1 UNKNOWN member, sk = -1) | Conformed airport dimension, hash surrogate key `airport_sk` |
| `dim_date` | calendar date | Calendar for flight dates and status-change dates |
| `dim_flight_status` | status code | Status groups and the cancellation flag used by every KPI |
| `fact_flight_activity` | flight_id | Current state of each flight + delay, fare, quality flags |
| `fact_status_transitions` | flight_id + source_sequence | Initial status and every CDC status change |
| `fact_secondary_events` | event_id | Auxiliary operational events linked to the flight |
| `mart_flight_daily` | flight_date | Network-wide departure delay, cancellation rate, on-time % |
| `mart_airport_daily` | flight_date + airport | **Airport punctuality** |
| `mart_airline_daily` | flight_date + airline | Delay attribution by airline |
| `gold_kpi_definitions` | kpi_id | Business definition + formula of every KPI |
| `gold_reconciliation` | batch_id + check_id | Expected-vs-actual evidence (TC07, TC08) |
| `gold_run_log` | batch_id + table | Inserted / updated rows per run (freshness, monitoring) |
| `gold_benchmark` | run | Before / after OPTIMIZE timings (TC10) |

**Expected results on your data** (these follow from the Silver numbers)

| Item | Expected |
|---|---|
| `dim_airline` / `dim_airport` | 111 / 151 (110 / 150 + UNKNOWN) |
| `dim_date` | 41 days (2026-07-01 to 2026-08-10) |
| `fact_flight_activity` | 11,500 (unmatched airline / airport = 0) |
| `fact_status_transitions` | 12,020 (11,500 INITIAL + 520 CDC) |
| `fact_secondary_events` | 310 |
| `mart_flight_daily` / `mart_airport_daily` / `mart_airline_daily` | 32 / 4,800 / 3,520 |
| Cancelled flights / cancellation rate | 171 / 1.49 % |
| Operated flights / on-time (delay <= 15 min) / on-time % | 11,329 / 2,001 / 17.66 % |
| Average departure delay (operated flights) | 44.95 min |
| Replay of the same input | 0 rows inserted, 0 rows updated |

In [ ]:
# =====================================================
# CAPSTONE 03 - DAY 3 : GOLD LAYER CONFIGURATION
# =====================================================
import datetime as dt
import time
import uuid
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = 'capstone_3flight'
BRONZE_SCHEMA = 'flight_schema'
SILVER_SCHEMA = 'silver_schema'
GOLD_SCHEMA = 'gold_schema'

# True  = drop the Gold tables this notebook creates and rebuild from a blank schema (first run / demo).
# False = incremental run: MERGE only new or changed rows (use this in the Workflows job).
RESET_GOLD = True
RUN_BENCHMARK = True            # Step I: time a KPI query before / after OPTIMIZE
BENCH_RAN = False               # set to True by Step I when the benchmark is recorded
STRICT_EXPECTED_COUNTS = True   # assert the capstone row counts; set False once new source batches arrive

# Job parameters (Databricks Workflows can override these; ignored outside Databricks)
try:
    dbutils.widgets.text('reset_gold', str(RESET_GOLD).lower())
    dbutils.widgets.text('run_benchmark', str(RUN_BENCHMARK).lower())
    RESET_GOLD = dbutils.widgets.get('reset_gold').strip().lower() == 'true'
    RUN_BENCHMARK = dbutils.widgets.get('run_benchmark').strip().lower() == 'true'
except NameError:
    pass

# Business rules (documented assumptions - need client sign-off)
ON_TIME_THRESHOLD_MIN = 15          # a flight is on time when departure delay <= 15 minutes (industry A15 convention)
CANCELLED_STATUSES = ['CANCELLED']  # REVERSED is NOT a cancellation (same decision as Silver)

# Reference tables (same as Silver)
AIRLINE_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.airline'
AIRPORT_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.airport'

# Silver source tables (same names as 02_silver_quality_rebuilt)
SILVER_FLIGHT_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_events'
SILVER_REJECTED_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records'
SILVER_STATUS_HISTORY = f'{CATALOG}.{SILVER_SCHEMA}.silver_status_history'
SILVER_FLIGHT_CURRENT = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_current'
SILVER_SECONDARY_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events'
SILVER_ASOF_TABLE = f'{CATALOG}.{SILVER_SCHEMA}.silver_flight_event_status_asof'

# Gold target tables
def gold(name):
    return f'{CATALOG}.{GOLD_SCHEMA}.{name}'

TARGETS = {
    'dim_airline': gold('dim_airline'),
    'dim_airport': gold('dim_airport'),
    'dim_date': gold('dim_date'),
    'dim_status': gold('dim_flight_status'),
    'fact_flight': gold('fact_flight_activity'),
    'fact_transitions': gold('fact_status_transitions'),
    'fact_secondary': gold('fact_secondary_events'),
    'mart_daily': gold('mart_flight_daily'),
    'mart_airport': gold('mart_airport_daily'),
    'mart_airline': gold('mart_airline_daily'),
    'run_log': gold('gold_run_log'),
}
KPI_DEFS_TABLE = gold('gold_kpi_definitions')
RECON_TABLE = gold('gold_reconciliation')
BENCH_TABLE = gold('gold_benchmark')
ALL_GOLD_TABLES = list(TARGETS.values()) + [KPI_DEFS_TABLE, RECON_TABLE, BENCH_TABLE]

spark.conf.set('spark.sql.session.timeZone', 'UTC')     # same as Silver: every timestamp is UTC
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}')

if RESET_GOLD:
    for tbl in ALL_GOLD_TABLES + [gold('tmp_' + t.split('.')[-1]) for t in TARGETS.values()]:
        spark.sql(f'DROP TABLE IF EXISTS {tbl}')
    print('Gold tables dropped - starting from a blank schema.')

# Verify the Silver inputs exist and have the expected row counts
expected_inputs = {SILVER_FLIGHT_TABLE: 11500, SILVER_FLIGHT_CURRENT: 11500, SILVER_STATUS_HISTORY: 12020,
                   SILVER_SECONDARY_TABLE: 310, SILVER_ASOF_TABLE: 11500, AIRLINE_TABLE: 110, AIRPORT_TABLE: 150}
for table, expected in expected_inputs.items():
    assert spark.catalog.tableExists(table), f'{table} is missing - run the Silver notebook first'
    actual = spark.table(table).count()
    print(table, ':', actual, 'rows', '(OK)' if actual == expected else f'(EXPECTED {expected})')
    if STRICT_EXPECTED_COUNTS:
        assert actual == expected, f'{table} has {actual} rows, expected {expected}'

BATCH_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S') + '_' + uuid.uuid4().hex[:6]
RUN_TS = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
print('Gold configuration verified | batch_id:', BATCH_ID, '| reset:', RESET_GOLD)

In [ ]:
# =====================================================
# KPI DEFINITIONS  (saved as a table - every KPI has a business definition and a formula)
# =====================================================
kpi_rows = [
    # kpi_id, kpi_name, business_definition, formula, grain, gold_table
    ('K01', 'Total flights', 'Number of unique flights scheduled on the flight date',
     'COUNT(flight_id)', 'flight_date [+ airport | airline]', 'mart_*_daily.total_flights'),
    ('K02', 'Cancelled flights', 'Flights whose latest valid status is CANCELLED (REVERSED is not a cancellation)',
     "SUM(is_cancelled) where is_cancelled = current_status IN ('CANCELLED')", 'flight_date [+ airport | airline]', 'mart_*_daily.cancelled_flights'),
    ('K03', 'Cancellation rate %', 'Share of scheduled flights that were cancelled',
     '100 * cancelled_flights / total_flights', 'flight_date [+ airport | airline]', 'mart_*_daily.cancellation_rate_pct'),
    ('K04', 'Average departure delay (min)', 'Mean departure delay of operated (not cancelled) flights; delay_minutes is treated as departure delay',
     'SUM(departure_delay_minutes of operated) / operated_flights', 'flight_date [+ airport | airline]', 'mart_*_daily.avg_departure_delay_minutes'),
    ('K05', 'On-time flights', f'Operated flights with departure delay <= {ON_TIME_THRESHOLD_MIN} minutes',
     f'SUM(is_operated AND departure_delay_minutes <= {ON_TIME_THRESHOLD_MIN})', 'flight_date [+ airport | airline]', 'mart_*_daily.on_time_flights'),
    ('K06', 'Airport punctuality (on-time %)', 'Share of operated flights at the airport that left on time',
     '100 * on_time_flights / operated_flights', 'flight_date + airport', 'mart_airport_daily.on_time_performance_pct'),
    ('K07', 'Delayed flight rate %', f'Share of operated flights delayed more than {ON_TIME_THRESHOLD_MIN} minutes',
     '100 * delayed_flights / operated_flights', 'flight_date [+ airport | airline]', 'mart_*_daily.delayed_rate_pct'),
    ('K08', 'Fare outlier flights', 'Flights whose fare is above the Silver outlier threshold (5,000) - kept, flagged',
     'SUM(is_fare_outlier)', 'flight_date [+ airport | airline]', 'mart_*_daily.fare_outlier_flights'),
]
kpi_df = spark.createDataFrame(kpi_rows, 'kpi_id string, kpi_name string, business_definition string, formula string, grain string, gold_column string')
(kpi_df.withColumn('on_time_threshold_min', F.lit(ON_TIME_THRESHOLD_MIN))
       .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(KPI_DEFS_TABLE))
print('KPI definitions saved:', KPI_DEFS_TABLE, '| KPIs:', spark.table(KPI_DEFS_TABLE).count())
display(spark.table(KPI_DEFS_TABLE).orderBy('kpi_id'))

In [ ]:
# =====================================================
# GOLD BUILD FUNCTIONS  (pure DataFrame logic - the load AND the tests use these)
# =====================================================
UNKNOWN_SK = -1
AUDIT_COLS = ['_gold_batch_id', '_gold_inserted_at', '_gold_updated_at']


def with_row_hash(df, exclude=()):
    """sha256 of every business column -> MERGE updates a row only when something really changed."""
    cols = [c for c in df.columns if c not in set(exclude) | set(AUDIT_COLS) | {'row_hash'}]
    return df.withColumn('row_hash', F.sha2(F.concat_ws('||', *[F.coalesce(F.col(c).cast('string'), F.lit('<NULL>')) for c in cols]), 256))


def date_key(col):
    return F.date_format(col, 'yyyyMMdd').cast('int')


# ---------------- dimensions ----------------
def build_reference_dim(ref_df, key, prefix):
    """Conformed SCD1 dimension. Surrogate key = xxhash64(business key): deterministic, identical on every rerun."""
    need = [key, 'display_name', 'region', 'active_flag', 'effective_date', 'risk_tier']
    missing = [c for c in need if c not in ref_df.columns]
    assert not missing, f'reference table is missing columns {missing}'
    w = Window.partitionBy(key).orderBy(F.col('effective_date').desc_nulls_last(), F.col(f'{prefix}_name').asc())
    d = (ref_df.select(F.trim(F.col(key)).alias(key), F.trim('display_name').alias(f'{prefix}_name'),
                       F.upper(F.trim('region')).alias('region'), F.upper(F.trim('active_flag')).alias('active_flag'),
                       F.col('effective_date').cast('date').alias('effective_date'), F.upper(F.trim('risk_tier')).alias('risk_tier'))
               .filter(F.col(key).isNotNull())
               .withColumn('_rn', F.row_number().over(w)).filter('_rn = 1').drop('_rn'))
    d = d.select(F.xxhash64(F.col(key)).alias(f'{prefix}_sk'), '*',
                 (F.col('active_flag') == 'Y').alias('is_active'), F.lit(False).alias('is_unknown_member'))
    unknown = spark.range(1).select(   # UNKNOWN member always exists, even if the reference is empty

        F.lit(UNKNOWN_SK).cast('bigint').alias(f'{prefix}_sk'), F.lit('UNKNOWN').alias(key),
        F.lit(f'Unknown {prefix}').alias(f'{prefix}_name'), F.lit('UNKNOWN').alias('region'), F.lit('N').alias('active_flag'),
        F.lit(None).cast('date').alias('effective_date'), F.lit('UNKNOWN').alias('risk_tier'),
        F.lit(False).alias('is_active'), F.lit(True).alias('is_unknown_member'))
    return with_row_hash(d.unionByName(unknown))


def build_dim_date(start_date, end_date):
    d = (spark.createDataFrame([(start_date, end_date)], 'start_date date, end_date date')
              .select(F.explode(F.sequence('start_date', 'end_date')).alias('calendar_date')))
    d = d.select(date_key('calendar_date').alias('date_key'), 'calendar_date',
                 F.year('calendar_date').alias('year'), F.quarter('calendar_date').alias('quarter'),
                 F.month('calendar_date').alias('month'), F.date_format('calendar_date', 'MMMM').alias('month_name'),
                 F.weekofyear('calendar_date').alias('week_of_year'), F.dayofmonth('calendar_date').alias('day_of_month'),
                 F.dayofweek('calendar_date').alias('day_of_week'), F.date_format('calendar_date', 'EEEE').alias('day_name'),
                 F.dayofweek('calendar_date').isin(1, 7).alias('is_weekend'))
    return with_row_hash(d)


def build_dim_status():
    rows = [('OPEN', 'ACTIVE', False, False, 'EVENT', 'Flight open'),
            ('PENDING', 'ACTIVE', False, False, 'EVENT', 'Flight pending'),
            ('COMPLETED', 'COMPLETED', False, True, 'EVENT/CDC', 'Flight completed'),
            ('FAILED', 'EXCEPTION', False, False, 'EVENT', 'Flight failed (source meaning not documented)'),
            ('CANCELLED', 'CANCELLED', True, False, 'CDC', 'Flight cancelled - counted in cancellation rate'),
            ('REVERSED', 'EXCEPTION', False, False, 'CDC', 'Reversed (source meaning not documented) - not a cancellation')]
    d = spark.createDataFrame(rows, 'status_code string, status_group string, is_cancelled boolean, is_completed boolean, '
                                    'status_origin string, description string')
    d = d.withColumn('is_cancelled', F.col('status_code').isin(CANCELLED_STATUSES))
    return with_row_hash(d)


# ---------------- facts ----------------
def build_fact_flight_activity(events, current, asof, secondary, dim_airline, dim_airport):
    """Grain: one row per flight_id (current state)."""
    e = events.select('flight_id', 'event_ts_utc', 'ingest_ts_utc', 'airline_id', 'airport_id', 'flight_type',
                      'flight_type_was_missing', 'aircraft_id', 'fare_amount', 'delay_minutes', 'is_fare_outlier',
                      'is_delay_outlier', 'timezone_is_valid', 'source_schema_version',
                      F.col('flight_status').alias('reported_status'), F.col('_source_file').alias('source_file'))
    c = current.select('flight_id', 'initial_status', 'current_status', 'status_updated_at', 'last_source_sequence', 'status_source')
    a = (asof.withColumn('_rn', F.row_number().over(Window.partitionBy('flight_id').orderBy(F.col('event_ts_utc').desc())))
             .filter('_rn = 1').select('flight_id', 'status_at_event_time'))
    s = secondary.groupBy('flight_id').agg(F.count('*').alias('secondary_event_count'),
                                           F.sum(F.when(F.col('event_type') == 'ALERT', 1).otherwise(0)).alias('alert_event_count'))
    da = dim_airline.filter(~F.col('is_unknown_member')).select('airline_id', 'airline_sk')
    dp = dim_airport.filter(~F.col('is_unknown_member')).select('airport_id', 'airport_sk')

    f = (e.join(c, 'flight_id', 'left').join(a, 'flight_id', 'left').join(s, 'flight_id', 'left')
          .join(F.broadcast(da), 'airline_id', 'left').join(F.broadcast(dp), 'airport_id', 'left'))
    cur_status = F.coalesce(F.col('current_status'), F.col('reported_status'))
    is_cancelled = cur_status.isin(CANCELLED_STATUSES)
    f = f.select(
        'flight_id',
        date_key(F.to_date('event_ts_utc')).alias('flight_date_key'),
        F.to_date('event_ts_utc').alias('flight_date'),
        F.coalesce('airline_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airline_sk'),
        F.coalesce('airport_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airport_sk'),
        'airline_id', 'airport_id', 'flight_type', 'aircraft_id',
        'event_ts_utc', 'ingest_ts_utc',
        ((F.unix_timestamp('ingest_ts_utc') - F.unix_timestamp('event_ts_utc')) / 60).cast('int').alias('ingest_lag_minutes'),
        'reported_status', 'status_at_event_time',
        F.coalesce('initial_status', 'reported_status').alias('initial_status'),
        cur_status.alias('current_status'),
        F.coalesce('status_updated_at', 'event_ts_utc').alias('status_updated_at'),
        F.coalesce('last_source_sequence', F.lit(0)).cast('bigint').alias('last_source_sequence'),
        F.coalesce('status_source', F.lit('EVENT')).alias('status_source'),
        'fare_amount',
        F.col('delay_minutes').alias('departure_delay_minutes'),
        is_cancelled.alias('is_cancelled'),
        (~is_cancelled).alias('is_operated'),
        ((~is_cancelled) & (F.col('delay_minutes') <= ON_TIME_THRESHOLD_MIN)).alias('is_on_time'),
        ((~is_cancelled) & (F.col('delay_minutes') > ON_TIME_THRESHOLD_MIN)).alias('is_delayed'),
        'is_fare_outlier', 'is_delay_outlier', 'flight_type_was_missing', 'timezone_is_valid',
        F.coalesce('secondary_event_count', F.lit(0)).cast('int').alias('secondary_event_count'),
        F.coalesce('alert_event_count', F.lit(0)).cast('int').alias('alert_event_count'),
        'source_schema_version', 'source_file')
    return with_row_hash(f)


def build_fact_status_transitions(history, events, current, dim_airline, dim_airport):
    """Grain: one row per flight_id + source_sequence (sequence 0 = INITIAL status from the flight event)."""
    h = history.select('flight_id', 'source_sequence', 'record_type', 'effective_ts_utc', 'previous_status', 'new_status',
                       'change_reason', 'is_noop', 'source_file')
    e = events.select('flight_id', 'event_ts_utc', 'airline_id', 'airport_id')
    c = current.select('flight_id', F.col('last_source_sequence').alias('_cur_seq'))
    da = dim_airline.filter(~F.col('is_unknown_member')).select('airline_id', 'airline_sk')
    dp = dim_airport.filter(~F.col('is_unknown_member')).select('airport_id', 'airport_sk')
    t = (h.join(e, 'flight_id', 'left').join(c, 'flight_id', 'left')
          .join(F.broadcast(da), 'airline_id', 'left').join(F.broadcast(dp), 'airport_id', 'left'))
    t = t.select(
        'flight_id', F.col('source_sequence').cast('bigint').alias('source_sequence'),
        F.col('record_type').alias('transition_type'),
        'effective_ts_utc', F.to_date('effective_ts_utc').alias('effective_date'),
        date_key(F.to_date('effective_ts_utc')).alias('effective_date_key'),
        date_key(F.to_date('event_ts_utc')).alias('flight_date_key'),
        F.coalesce('airline_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airline_sk'),
        F.coalesce('airport_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airport_sk'),
        'previous_status', 'new_status', 'change_reason', 'is_noop',
        ((F.col('record_type') == 'CDC') & (~F.col('is_noop'))).alias('is_status_change'),
        ((F.col('record_type') == 'CDC') & F.col('new_status').isin(CANCELLED_STATUSES)).alias('is_cancellation'),
        (F.col('source_sequence') == F.col('_cur_seq')).alias('is_current_state'),
        ((F.unix_timestamp('effective_ts_utc') - F.unix_timestamp('event_ts_utc')) / 60).cast('bigint').alias('minutes_after_flight_event'),
        'source_file')
    return with_row_hash(t)


def build_fact_secondary_events(secondary, events, dim_airline, dim_airport):
    """Grain: one row per event_id."""
    e = events.select('flight_id', 'airline_id', 'airport_id')
    da = dim_airline.filter(~F.col('is_unknown_member')).select('airline_id', 'airline_sk')
    dp = dim_airport.filter(~F.col('is_unknown_member')).select('airport_id', 'airport_sk')
    s = (secondary.join(e, 'flight_id', 'left').join(F.broadcast(da), 'airline_id', 'left').join(F.broadcast(dp), 'airport_id', 'left'))
    s = s.select('event_id', 'flight_id', 'event_ts_utc', F.to_date('event_ts_utc').alias('event_date'),
                 date_key(F.to_date('event_ts_utc')).alias('event_date_key'),
                 F.coalesce('airline_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airline_sk'),
                 F.coalesce('airport_sk', F.lit(UNKNOWN_SK)).cast('bigint').alias('airport_sk'),
                 'event_type', F.col('event_value').cast('decimal(12,2)').alias('event_value'), 'related_entity_id',
                 F.regexp_extract('related_entity_id', '^([A-Z]+)', 1).alias('related_entity_type'),
                 F.col('_source_file').alias('source_file'))
    return with_row_hash(s)


# ---------------- KPI marts ----------------
def kpi_aggregate(fact_df, group_cols):
    """Additive measures first (they reconcile exactly); ratios are derived from them."""
    op = F.col('is_operated')
    agg = fact_df.groupBy(*group_cols).agg(
        F.count('*').cast('bigint').alias('total_flights'),
        F.sum(F.col('is_cancelled').cast('int')).cast('bigint').alias('cancelled_flights'),
        F.sum(op.cast('int')).cast('bigint').alias('operated_flights'),
        F.sum(F.col('is_on_time').cast('int')).cast('bigint').alias('on_time_flights'),
        F.sum(F.col('is_delayed').cast('int')).cast('bigint').alias('delayed_flights'),
        F.coalesce(F.sum(F.when(op, F.col('departure_delay_minutes'))), F.lit(0)).cast('bigint').alias('total_departure_delay_minutes'),
        F.max(F.when(op, F.col('departure_delay_minutes'))).alias('max_departure_delay_minutes'),
        F.sum(F.when(F.col('current_status') == 'REVERSED', 1).otherwise(0)).cast('bigint').alias('reversed_flights'),
        F.sum('fare_amount').cast('decimal(18,2)').alias('total_fare_amount'),
        F.sum(F.col('is_fare_outlier').cast('int')).cast('bigint').alias('fare_outlier_flights'),
        F.sum('secondary_event_count').cast('bigint').alias('secondary_event_count'))

    def pct(n, d):
        return F.when(F.col(d) > 0, F.round(100.0 * F.col(n) / F.col(d), 2))
    return (agg.withColumn('cancellation_rate_pct', pct('cancelled_flights', 'total_flights'))
               .withColumn('on_time_performance_pct', pct('on_time_flights', 'operated_flights'))
               .withColumn('delayed_rate_pct', pct('delayed_flights', 'operated_flights'))
               .withColumn('avg_departure_delay_minutes',
                           F.when(F.col('operated_flights') > 0,
                                  F.round(F.col('total_departure_delay_minutes') / F.col('operated_flights'), 2))))


def build_marts(fact_df, dim_airline, dim_airport, dim_date, dates=None):
    """Recompute the three marts for the given flight dates (None = every date)."""
    f = fact_df if dates is None else fact_df.filter(F.col('flight_date').isin(list(dates)))
    dd = dim_date.select(F.col('date_key').alias('flight_date_key'), 'day_name', 'is_weekend')
    daily = kpi_aggregate(f, ['flight_date', 'flight_date_key']).join(dd, 'flight_date_key', 'left')
    airport = (kpi_aggregate(f, ['flight_date', 'flight_date_key', 'airport_sk'])
               .join(dim_airport.select('airport_sk', 'airport_id', 'airport_name', F.col('region').alias('airport_region')), 'airport_sk', 'left'))
    airline = (kpi_aggregate(f, ['flight_date', 'flight_date_key', 'airline_sk'])
               .join(dim_airline.select('airline_sk', 'airline_id', 'airline_name', F.col('region').alias('airline_region')), 'airline_sk', 'left'))
    return with_row_hash(daily), with_row_hash(airport), with_row_hash(airline)


print('Gold build functions defined.')

In [ ]:
# =====================================================
# DELTA LOAD HELPERS  (idempotent MERGE - Databricks only)
# =====================================================
def merge_upsert(target, src_df, key_cols, batch_id, run_ts, delete_dates=None):
    """Insert new keys, update a key only when its row_hash changed, leave unchanged rows alone.
    delete_dates (marts only): keys inside these flight dates that no longer exist in the source are deleted.
    Returns {'inserted': n, 'updated': n} for this batch."""
    run_ts_lit = F.lit(run_ts).cast('timestamp')
    src = (src_df.withColumn('_gold_batch_id', F.lit(batch_id))
                 .withColumn('_gold_inserted_at', run_ts_lit)
                 .withColumn('_gold_updated_at', run_ts_lit))
    if not spark.catalog.tableExists(target):
        src.limit(0).write.format('delta').saveAsTable(target)
        spark.sql(f"ALTER TABLE {target} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
    view = '_src_' + target.split('.')[-1]
    src.createOrReplaceTempView(view)
    cols = src.columns
    on = ' AND '.join(f't.`{k}` = s.`{k}`' for k in key_cols)
    upd = ', '.join(f't.`{c}` = s.`{c}`' for c in cols if c != '_gold_inserted_at')
    ins_cols = ', '.join(f'`{c}`' for c in cols)
    ins_vals = ', '.join(f's.`{c}`' for c in cols)
    sql = (f'MERGE INTO {target} t USING {view} s ON {on} '
           f'WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET {upd} '
           f'WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})')
    if delete_dates:
        in_list = ', '.join(f"DATE'{d}'" for d in sorted(delete_dates))
        sql += f' WHEN NOT MATCHED BY SOURCE AND t.flight_date IN ({in_list}) THEN DELETE'
    spark.sql(sql)
    changed = spark.table(target).filter(F.col('_gold_batch_id') == batch_id)
    return {'inserted': changed.filter(F.col('_gold_inserted_at') == run_ts_lit).count(),
            'updated': changed.filter(F.col('_gold_inserted_at') < run_ts_lit).count()}


def load_sources():
    return {'events': spark.table(SILVER_FLIGHT_TABLE), 'current': spark.table(SILVER_FLIGHT_CURRENT),
            'history': spark.table(SILVER_STATUS_HISTORY), 'secondary': spark.table(SILVER_SECONDARY_TABLE),
            'asof': spark.table(SILVER_ASOF_TABLE), 'airline': spark.table(AIRLINE_TABLE), 'airport': spark.table(AIRPORT_TABLE)}


def run_gold_load(targets, src, batch_id, run_ts, full_refresh_marts=False):
    """The whole Gold load. Re-running it on unchanged input inserts and updates nothing."""
    log = []

    def step(name, df, keys, delete_dates=None):
        m = merge_upsert(targets[name], df, keys, batch_id, run_ts, delete_dates)
        log.append((batch_id, run_ts, targets[name].split('.')[-1], m['inserted'], m['updated'],
                    spark.table(targets[name]).count()))
        return m

    # 1. dimensions
    m_al = step('dim_airline', build_reference_dim(src['airline'], 'airline_id', 'airline'), ['airline_id'])
    m_ap = step('dim_airport', build_reference_dim(src['airport'], 'airport_id', 'airport'), ['airport_id'])
    bounds = (src['events'].select(F.to_date('event_ts_utc').alias('d'))
              .unionByName(src['history'].select(F.to_date('effective_ts_utc').alias('d')))
              .agg(F.min('d').alias('lo'), F.max('d').alias('hi')).first())
    step('dim_date', build_dim_date(bounds.lo, bounds.hi), ['date_key'])
    step('dim_status', build_dim_status(), ['status_code'])
    dim_airline, dim_airport = spark.table(targets['dim_airline']), spark.table(targets['dim_airport'])
    dim_date = spark.table(targets['dim_date'])

    # 2. facts
    step('fact_flight', build_fact_flight_activity(src['events'], src['current'], src['asof'], src['secondary'],
                                                   dim_airline, dim_airport), ['flight_id'])
    step('fact_transitions', build_fact_status_transitions(src['history'], src['events'], src['current'],
                                                           dim_airline, dim_airport), ['flight_id', 'source_sequence'])
    step('fact_secondary', build_fact_secondary_events(src['secondary'], src['events'], dim_airline, dim_airport), ['event_id'])

    # 3. marts - recompute only the flight dates touched by this batch (all dates if a dimension attribute changed)
    fact = spark.table(targets['fact_flight'])
    dims_changed = m_al['updated'] + m_ap['updated'] > 0
    if full_refresh_marts or dims_changed or not spark.catalog.tableExists(targets['mart_daily']):
        dates = [r[0] for r in fact.select('flight_date').distinct().collect()]
    else:
        dates = [r[0] for r in fact.filter(F.col('_gold_batch_id') == batch_id).select('flight_date').distinct().collect()]
    if dates:
        daily, airport, airline = build_marts(fact, dim_airline, dim_airport, dim_date, dates)
        step('mart_daily', daily, ['flight_date'], dates)
        step('mart_airport', airport, ['flight_date', 'airport_sk'], dates)
        step('mart_airline', airline, ['flight_date', 'airline_sk'], dates)
    else:
        for name in ['mart_daily', 'mart_airport', 'mart_airline']:
            log.append((batch_id, run_ts, targets[name].split('.')[-1], 0, 0, spark.table(targets[name]).count()))

    # 4. run log (freshness + monitoring)
    log_df = spark.createDataFrame(log, 'batch_id string, run_ts timestamp, table_name string, rows_inserted long, '
                                        'rows_updated long, target_rows long')
    log_df = log_df.withColumn('affected_flight_dates', F.lit(len(dates)))
    log_df.write.format('delta').mode('append').saveAsTable(targets['run_log'])
    return log_df


print('Load helpers defined.')

## Step A - Run the Gold load (dimensions -> facts -> marts)

In [ ]:
sources = load_sources()
run_metrics = run_gold_load(TARGETS, sources, BATCH_ID, RUN_TS)
display(run_metrics)

print('Gold table row counts')
for key, name in TARGETS.items():
    print(f'{name.split(".")[-1]:28s}', spark.table(name).count())

## Step B - Dimensions: keys, unknown member and join coverage (TC07)

In [ ]:
dim_airline = spark.table(TARGETS['dim_airline'])
dim_airport = spark.table(TARGETS['dim_airport'])
dim_date = spark.table(TARGETS['dim_date'])
fact = spark.table(TARGETS['fact_flight'])

for name, d, sk in [('dim_airline', dim_airline, 'airline_sk'), ('dim_airport', dim_airport, 'airport_sk')]:
    n, n_sk = d.count(), d.select(sk).distinct().count()
    print(f'{name}: rows {n} | distinct {sk} {n_sk} | unknown member rows {d.filter("is_unknown_member").count()}')
    assert n == n_sk, f'{name}: surrogate key is not unique'

print('dim_date:', dim_date.agg(F.min('calendar_date'), F.max('calendar_date'), F.count('*')).first())

# TC07 - every fact row must find its dimension row (UNKNOWN = -1 means an unmatched business key)
unmatched = fact.agg(F.sum((F.col('airline_sk') == UNKNOWN_SK).cast('int')).alias('unmatched_airline'),
                     F.sum((F.col('airport_sk') == UNKNOWN_SK).cast('int')).alias('unmatched_airport')).first()
print('Unmatched airline keys:', unmatched.unmatched_airline, '| unmatched airport keys:', unmatched.unmatched_airport)

print('Flights on inactive airlines / airports (observation, not an error)')
display(fact.join(dim_airline.select('airline_sk', F.col('is_active').alias('airline_active')), 'airline_sk')
            .join(dim_airport.select('airport_sk', F.col('is_active').alias('airport_active')), 'airport_sk')
            .agg(F.sum((~F.col('airline_active')).cast('int')).alias('flights_on_inactive_airline'),
                 F.sum((~F.col('airport_active')).cast('int')).alias('flights_on_inactive_airport')))
display(dim_airline.orderBy('airline_id').limit(5))

## Step C - Facts: grain, status mix and one flight traced Silver -> Gold

In [ ]:
fact = spark.table(TARGETS['fact_flight'])
trans = spark.table(TARGETS['fact_transitions'])
sec_f = spark.table(TARGETS['fact_secondary'])

print('fact_flight_activity rows:', fact.count(), '| distinct flight_id:', fact.select('flight_id').distinct().count())
print('fact_status_transitions rows:', trans.count(), '| distinct (flight_id, source_sequence):',
      trans.select('flight_id', 'source_sequence').distinct().count())
print('fact_secondary_events rows:', sec_f.count(), '| distinct event_id:', sec_f.select('event_id').distinct().count())

print('Current status mix (fact) - must equal silver_flight_current')
display(fact.groupBy('current_status', 'is_cancelled').count().orderBy('current_status'))
print('Transitions by type')
display(trans.groupBy('transition_type', 'is_status_change', 'is_cancellation').count().orderBy('transition_type'))

fid = 'FLIGHT-0000001'
print(f'Trace {fid}: Silver current -> Silver history -> Gold fact -> Gold transitions')
display(spark.table(SILVER_FLIGHT_CURRENT).filter(F.col('flight_id') == fid))
display(spark.table(SILVER_STATUS_HISTORY).filter(F.col('flight_id') == fid).orderBy('source_sequence'))
display(fact.filter(F.col('flight_id') == fid).select('flight_id', 'flight_date', 'airline_sk', 'airport_sk', 'reported_status',
        'status_at_event_time', 'current_status', 'status_source', 'departure_delay_minutes', 'is_cancelled', 'is_on_time'))
display(trans.filter(F.col('flight_id') == fid).orderBy('source_sequence'))

## Step D - KPI marts: departure delay, cancellation rate, airport punctuality

In [ ]:
mart_daily = spark.table(TARGETS['mart_daily'])
mart_airport = spark.table(TARGETS['mart_airport'])
mart_airline = spark.table(TARGETS['mart_airline'])
print('mart_flight_daily:', mart_daily.count(), '| mart_airport_daily:', mart_airport.count(), '| mart_airline_daily:', mart_airline.count())

print('Daily network KPIs')
display(mart_daily.select('flight_date', 'day_name', 'total_flights', 'cancelled_flights', 'cancellation_rate_pct',
                          'operated_flights', 'on_time_flights', 'on_time_performance_pct', 'avg_departure_delay_minutes')
                  .orderBy('flight_date'))

print('Whole-period KPIs (re-derived from the additive measures)')
overall = mart_daily.agg(F.sum('total_flights').alias('total_flights'), F.sum('cancelled_flights').alias('cancelled'),
                         F.sum('operated_flights').alias('operated'), F.sum('on_time_flights').alias('on_time'),
                         F.sum('total_departure_delay_minutes').alias('delay_min'))
overall = (overall.withColumn('cancellation_rate_pct', F.round(100 * F.col('cancelled') / F.col('total_flights'), 2))
                  .withColumn('on_time_performance_pct', F.round(100 * F.col('on_time') / F.col('operated'), 2))
                  .withColumn('avg_departure_delay_minutes', F.round(F.col('delay_min') / F.col('operated'), 2)))
display(overall)

print('Least punctual airports over the period (min 50 operated flights)')
display(mart_airport.groupBy('airport_id', 'airport_name', 'airport_region')
        .agg(F.sum('operated_flights').alias('operated'), F.sum('on_time_flights').alias('on_time'),
             F.sum('cancelled_flights').alias('cancelled'), F.sum('total_departure_delay_minutes').alias('delay_min'))
        .filter('operated >= 50')
        .withColumn('on_time_performance_pct', F.round(100 * F.col('on_time') / F.col('operated'), 2))
        .withColumn('avg_departure_delay_minutes', F.round(F.col('delay_min') / F.col('operated'), 2))
        .orderBy('on_time_performance_pct', 'airport_id').limit(10))

print('Delay attribution by airline (top 10 by total delay minutes)')
display(mart_airline.groupBy('airline_id', 'airline_name')
        .agg(F.sum('total_departure_delay_minutes').alias('delay_min'), F.sum('operated_flights').alias('operated'))
        .withColumn('avg_departure_delay_minutes', F.round(F.col('delay_min') / F.col('operated'), 2))
        .orderBy(F.col('delay_min').desc(), 'airline_id').limit(10))

## Step E - Reconciliation: Silver vs Gold, expected vs actual (TC07, TC08)

In [ ]:
# Expected values are computed INDEPENDENTLY from Silver with plain SQL (not with the Gold functions).
T = ON_TIME_THRESHOLD_MIN
cancel_list = ', '.join(f"'{s}'" for s in CANCELLED_STATUSES)
silver_flat = f"""
    SELECT e.flight_id, to_date(e.event_ts_utc) AS flight_date, e.airline_id, e.airport_id, e.fare_amount, e.delay_minutes,
           c.current_status, c.current_status IN ({cancel_list}) AS cancelled
    FROM {SILVER_FLIGHT_TABLE} e JOIN {SILVER_FLIGHT_CURRENT} c ON e.flight_id = c.flight_id"""
exp = spark.sql(f"""
    SELECT COUNT(*) AS flights, SUM(CAST(cancelled AS INT)) AS cancelled, SUM(CAST(NOT cancelled AS INT)) AS operated,
           SUM(CASE WHEN NOT cancelled AND delay_minutes <= {T} THEN 1 ELSE 0 END) AS on_time,
           SUM(CASE WHEN NOT cancelled THEN delay_minutes ELSE 0 END) AS delay_operated,
           SUM(delay_minutes) AS delay_all, CAST(SUM(fare_amount) AS DECIMAL(18,2)) AS fare
    FROM ({silver_flat})""").first()

fact = spark.table(TARGETS['fact_flight'])
trans = spark.table(TARGETS['fact_transitions'])
mart_daily, mart_airport, mart_airline = (spark.table(TARGETS[k]) for k in ['mart_daily', 'mart_airport', 'mart_airline'])
dim_airline, dim_airport, dim_date = (spark.table(TARGETS[k]) for k in ['dim_airline', 'dim_airport', 'dim_date'])
hist = spark.table(SILVER_STATUS_HISTORY)

act = fact.agg(F.count('*').alias('flights'), F.sum(F.col('is_cancelled').cast('int')).alias('cancelled'),
               F.sum(F.col('is_operated').cast('int')).alias('operated'), F.sum(F.col('is_on_time').cast('int')).alias('on_time'),
               F.sum(F.when(F.col('is_operated'), F.col('departure_delay_minutes')).otherwise(0)).alias('delay_operated'),
               F.sum('departure_delay_minutes').alias('delay_all'), F.sum('fare_amount').cast('decimal(18,2)').alias('fare')).first()


def mart_sums(m):
    return m.agg(*[F.sum(c).alias(c) for c in ['total_flights', 'cancelled_flights', 'operated_flights', 'on_time_flights',
                                               'total_departure_delay_minutes']],
                 F.sum('total_fare_amount').cast('decimal(18,2)').alias('fare')).first()


md_, ma_, ml_ = mart_sums(mart_daily), mart_sums(mart_airport), mart_sums(mart_airline)

# TC08 - independent per-date KPI query vs mart_flight_daily (additive measures exact, ratios to 0.01)
indep = spark.sql(f"""
    SELECT flight_date, COUNT(*) AS total_flights, SUM(CAST(cancelled AS INT)) AS cancelled_flights,
           SUM(CASE WHEN NOT cancelled AND delay_minutes <= {T} THEN 1 ELSE 0 END) AS on_time_flights,
           ROUND(100.0 * SUM(CAST(cancelled AS INT)) / COUNT(*), 2) AS cancellation_rate_pct,
           ROUND(100.0 * SUM(CASE WHEN NOT cancelled AND delay_minutes <= {T} THEN 1 ELSE 0 END)
                 / SUM(CAST(NOT cancelled AS INT)), 2) AS on_time_performance_pct,
           ROUND(SUM(CASE WHEN NOT cancelled THEN delay_minutes ELSE 0 END) / SUM(CAST(NOT cancelled AS INT)), 2) AS avg_departure_delay_minutes
    FROM ({silver_flat}) GROUP BY flight_date""")
cmp = indep.alias('i').join(mart_daily.alias('m'), 'flight_date', 'full_outer')
date_mismatch = cmp.filter(
    F.col('m.total_flights').isNull() | F.col('i.total_flights').isNull()
    | (F.col('i.total_flights') != F.col('m.total_flights')) | (F.col('i.cancelled_flights') != F.col('m.cancelled_flights'))
    | (F.col('i.on_time_flights') != F.col('m.on_time_flights'))
    | (F.abs(F.col('i.cancellation_rate_pct') - F.col('m.cancellation_rate_pct')) > 0.01)
    | (F.abs(F.col('i.on_time_performance_pct') - F.col('m.on_time_performance_pct')) > 0.01)
    | (F.abs(F.col('i.avg_departure_delay_minutes') - F.col('m.avg_departure_delay_minutes')) > 0.01)).count()

# TC08 - independent per-airport punctuality vs mart_airport_daily
indep_ap = spark.sql(f"""
    SELECT flight_date, airport_id, COUNT(*) AS total_flights,
           SUM(CASE WHEN NOT cancelled AND delay_minutes <= {T} THEN 1 ELSE 0 END) AS on_time_flights
    FROM ({silver_flat}) GROUP BY flight_date, airport_id""")
ap_mismatch = (indep_ap.alias('i').join(mart_airport.alias('m'), ['flight_date', 'airport_id'], 'full_outer')
               .filter(F.col('m.total_flights').isNull() | F.col('i.total_flights').isNull()
                       | (F.col('i.total_flights') != F.col('m.total_flights'))
                       | (F.col('i.on_time_flights') != F.col('m.on_time_flights'))).count())

fact_dates = fact.select(F.col('flight_date_key').alias('date_key')).union(trans.select(F.col('effective_date_key').alias('date_key'))).distinct()
missing_dates = fact_dates.join(dim_date, 'date_key', 'left_anti').count()
unknown_status = fact.select(F.col('current_status').alias('status_code')).distinct().join(
    spark.table(TARGETS['dim_status']), 'status_code', 'left_anti').count()

checks = [
    # check_id, test_id, description, expected, actual
    ('R01', 'TC08', 'fact rows = silver_flight_current rows', spark.table(SILVER_FLIGHT_CURRENT).count(), act.flights),
    ('R02', 'TC08', 'fact rows = silver_flight_events rows', spark.table(SILVER_FLIGHT_TABLE).count(), act.flights),
    ('R03', 'TC03', 'fact flight_id is unique', act.flights, fact.select('flight_id').distinct().count()),
    ('R04', 'TC08', 'sum departure delay (all flights) Silver = Gold', exp.delay_all, act.delay_all),
    ('R05', 'TC08', 'sum fare Silver = Gold', str(exp.fare), str(act.fare)),
    ('R06', 'TC08', 'cancelled flights Silver = Gold', exp.cancelled, act.cancelled),
    ('R07', 'TC08', 'on-time flights Silver = Gold', exp.on_time, act.on_time),
    ('R08', 'TC07', 'fact rows with unmatched airline (sk = -1)', 0, fact.filter(F.col('airline_sk') == UNKNOWN_SK).count()),
    ('R09', 'TC07', 'fact rows with unmatched airport (sk = -1)', 0, fact.filter(F.col('airport_sk') == UNKNOWN_SK).count()),
    ('R10', 'TC07', 'dim_airline business rows = reference rows', spark.table(AIRLINE_TABLE).count(), dim_airline.filter('NOT is_unknown_member').count()),
    ('R11', 'TC07', 'dim_airport business rows = reference rows', spark.table(AIRPORT_TABLE).count(), dim_airport.filter('NOT is_unknown_member').count()),
    ('R12', 'TC07', 'fact / transition dates missing from dim_date', 0, missing_dates),
    ('R13', 'TC07', 'fact statuses missing from dim_flight_status', 0, unknown_status),
    ('R14', 'TC04', 'transitions rows = silver_status_history rows', hist.count(), trans.count()),
    ('R15', 'TC04', 'CDC transitions = silver CDC history rows', hist.filter("record_type = 'CDC'").count(), trans.filter("transition_type = 'CDC'").count()),
    ('R16', 'TC04', 'transitions marked current = flights', act.flights, trans.filter('is_current_state').count()),
    ('R17', 'TC08', 'fact_secondary_events = silver_secondary_events', spark.table(SILVER_SECONDARY_TABLE).count(), spark.table(TARGETS['fact_secondary']).count()),
    ('R18', 'TC08', 'mart_flight_daily total flights = fact', act.flights, md_.total_flights),
    ('R19', 'TC08', 'mart_flight_daily cancelled = fact', act.cancelled, md_.cancelled_flights),
    ('R20', 'TC08', 'mart_flight_daily operated = fact', act.operated, md_.operated_flights),
    ('R21', 'TC08', 'mart_flight_daily on-time = fact', act.on_time, md_.on_time_flights),
    ('R22', 'TC08', 'mart_flight_daily delay minutes = fact (operated)', act.delay_operated, md_.total_departure_delay_minutes),
    ('R23', 'TC08', 'mart_flight_daily fare = fact', str(act.fare), str(md_.fare)),
    ('R24', 'TC08', 'mart_airport_daily totals = fact (flights|cancelled|on-time|delay)',
     f'{act.flights}|{act.cancelled}|{act.on_time}|{act.delay_operated}',
     f'{ma_.total_flights}|{ma_.cancelled_flights}|{ma_.on_time_flights}|{ma_.total_departure_delay_minutes}'),
    ('R25', 'TC08', 'mart_airline_daily totals = fact (flights|cancelled|on-time|delay)',
     f'{act.flights}|{act.cancelled}|{act.on_time}|{act.delay_operated}',
     f'{ml_.total_flights}|{ml_.cancelled_flights}|{ml_.on_time_flights}|{ml_.total_departure_delay_minutes}'),
    ('R26', 'TC08', 'mart_flight_daily one row per flight_date', mart_daily.count(), mart_daily.select('flight_date').distinct().count()),
    ('R27', 'TC08', 'mart_airport_daily one row per date + airport', mart_airport.count(), mart_airport.select('flight_date', 'airport_sk').distinct().count()),
    ('R28', 'TC08', 'mart_airline_daily one row per date + airline', mart_airline.count(), mart_airline.select('flight_date', 'airline_sk').distinct().count()),
    ('R29', 'TC08', 'independent daily KPI query vs mart_flight_daily: mismatching dates', 0, date_mismatch),
    ('R30', 'TC08', 'independent airport punctuality query vs mart_airport_daily: mismatching rows', 0, ap_mismatch),
]
if STRICT_EXPECTED_COUNTS:   # capstone constants (from the Silver expected results)
    checks += [('R31', 'TC01', 'capstone: 11,500 flights in Gold', 11500, act.flights),
               ('R32', 'TC04', 'capstone: 171 cancelled flights', 171, act.cancelled),
               ('R33', 'TC08', 'capstone: overall cancellation rate %', '1.49', f'{100 * act.cancelled / act.flights:.2f}'),
               ('R34', 'TC08', 'capstone: overall on-time performance %', '17.66', f'{100 * act.on_time / act.operated:.2f}'),
               ('R35', 'TC08', 'capstone: average departure delay (operated)', '44.95', f'{act.delay_operated / act.operated:.2f}')]

recon = spark.createDataFrame([(BATCH_ID, c[0], c[1], c[2], str(c[3]), str(c[4]), 'PASS' if str(c[3]) == str(c[4]) else 'FAIL')
                               for c in checks],
                              'batch_id string, check_id string, test_id string, description string, expected string, actual string, result string')
recon = recon.withColumn('checked_at', F.current_timestamp())
recon.write.format('delta').mode('append').option('mergeSchema', 'true').saveAsTable(RECON_TABLE)
display(spark.table(RECON_TABLE).filter(F.col('batch_id') == BATCH_ID).orderBy('check_id'))

recon_failed = recon.filter("result = 'FAIL'").count()
print('Reconciliation checks:', len(checks), '| failed:', recon_failed)
assert recon_failed == 0, 'reconciliation failed - see gold_reconciliation'

## Step F - Replay test: running the same load again must change nothing (TC03, TC06)

In [ ]:
def kpi_snapshot():
    return {k: sorted(str(r) for r in spark.table(TARGETS[k]).drop(*AUDIT_COLS).collect())
            for k in ['mart_daily', 'mart_airport', 'mart_airline']}

before = kpi_snapshot()
before_counts = {k: spark.table(v).count() for k, v in TARGETS.items() if k != 'run_log'}

REPLAY_BATCH = BATCH_ID + '_replay'
REPLAY_TS = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
replay = run_gold_load(TARGETS, load_sources(), REPLAY_BATCH, REPLAY_TS)
display(replay)

after = kpi_snapshot()
after_counts = {k: spark.table(v).count() for k, v in TARGETS.items() if k != 'run_log'}
changed_rows = replay.agg(F.sum('rows_inserted') + F.sum('rows_updated')).first()[0]
print('rows inserted + updated by the replay:', changed_rows)
print('table counts unchanged:', before_counts == after_counts)
print('KPI marts unchanged:', before == after)
assert changed_rows == 0 and before_counts == after_counts and before == after
print('PASS - replaying the same input does not double count')

## Step G - Incremental test on temporary tables (later status revision + new flight)

Uses the **same** `run_gold_load` function on `tmp_*` copies, so the real Gold tables are not touched. A simulated batch cancels one OPEN flight and adds one new flight on the same date. Only that one date may change in the marts.

In [ ]:
TMP = {k: gold('tmp_' + v.split('.')[-1]) for k, v in TARGETS.items()}
inc_results = []

def inc_check(test_id, description, condition):
    inc_results.append((test_id, description, 'PASS' if condition else 'FAIL'))

try:
    for t in TMP.values():
        spark.sql(f'DROP TABLE IF EXISTS {t}')
    base = load_sources()
    ts1 = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    run_gold_load(TMP, base, 'inc_1', ts1)
    daily_1 = {r.flight_date: r for r in spark.table(TMP['mart_daily']).collect()}

    # simulated batch: flight X (currently OPEN) is cancelled later; a new flight is added on X's date
    x = (spark.table(TMP['fact_flight']).filter("current_status = 'OPEN'").orderBy('flight_id').first())
    new_id = 'FLIGHT-9000001'
    change_ts = dt.datetime(2026, 8, 11, 0, 0, 0)
    ev = base['events']
    cur = base['current']
    hist = base['history']
    sim = {
        'events': ev.unionByName(ev.filter(F.col('flight_id') == x.flight_id)
                                   .withColumn('flight_id', F.lit(new_id)).withColumn('_source_file', F.lit('simulated_batch.csv'))),
        'current': (cur.withColumn('current_status', F.when(F.col('flight_id') == x.flight_id, F.lit('CANCELLED')).otherwise(F.col('current_status')))
                       .withColumn('status_updated_at', F.when(F.col('flight_id') == x.flight_id, F.lit(change_ts)).otherwise(F.col('status_updated_at')))
                       .withColumn('last_source_sequence', F.when(F.col('flight_id') == x.flight_id, F.lit(9999).cast('bigint')).otherwise(F.col('last_source_sequence')))
                       .withColumn('status_source', F.when(F.col('flight_id') == x.flight_id, F.lit('CDC')).otherwise(F.col('status_source')))
                       .unionByName(cur.filter(F.col('flight_id') == x.flight_id).withColumn('flight_id', F.lit(new_id)))),
        'history': (hist.unionByName(hist.filter((F.col('flight_id') == x.flight_id) & (F.col('source_sequence') == 0))
                                         .withColumn('flight_id', F.lit(new_id)))
                        .unionByName(hist.filter((F.col('flight_id') == x.flight_id) & (F.col('source_sequence') == 0))
                                         .withColumn('effective_ts_utc', F.lit(change_ts)).withColumn('previous_status', F.lit('OPEN'))
                                         .withColumn('new_status', F.lit('CANCELLED')).withColumn('change_reason', F.lit('SIMULATED_TEST'))
                                         .withColumn('source_sequence', F.lit(9999).cast('bigint')).withColumn('record_type', F.lit('CDC'))
                                         .withColumn('source_file', F.lit('simulated_cdc.csv')))),
        'secondary': base['secondary'],
        'asof': base['asof'].unionByName(base['asof'].filter(F.col('flight_id') == x.flight_id).withColumn('flight_id', F.lit(new_id))),
        'airline': base['airline'], 'airport': base['airport']}

    ts2 = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    m2 = {r.table_name: r for r in run_gold_load(TMP, sim, 'inc_2', ts2).collect()}
    ff = m2['tmp_fact_flight_activity']
    tr = m2['tmp_fact_status_transitions']
    md2 = m2['tmp_mart_flight_daily']
    inc_check('G01', 'fact: exactly 1 inserted (new flight) and 1 updated (cancelled flight)', ff.rows_inserted == 1 and ff.rows_updated == 1)
    inc_check('G02', 'transitions: 2 inserted (new CDC row + new flight INITIAL), 1 updated (old current flag)',
              tr.rows_inserted == 2 and tr.rows_updated == 1)
    inc_check('G03', 'marts: only 1 flight date recomputed', md2.affected_flight_dates == 1 and md2.rows_updated == 1 and md2.rows_inserted == 0)
    daily_2 = {r.flight_date: r for r in spark.table(TMP['mart_daily']).collect()}
    d = x.flight_date
    inc_check('G04', 'that date: total_flights +1 and cancelled_flights +1',
              daily_2[d].total_flights == daily_1[d].total_flights + 1 and daily_2[d].cancelled_flights == daily_1[d].cancelled_flights + 1)
    strip = lambda r: {k: v for k, v in r.asDict().items() if k not in AUDIT_COLS}
    inc_check('G05', 'every other date is unchanged', all(strip(daily_2[k]) == strip(daily_1[k]) for k in daily_1 if k != d))
    inc_check('G06', 'late status is the new current state in the fact',
              spark.table(TMP['fact_flight']).filter(F.col('flight_id') == x.flight_id).first().current_status == 'CANCELLED')

    ts3 = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    m3 = run_gold_load(TMP, sim, 'inc_3', ts3)
    inc_check('G07', 'replaying the simulated batch changes nothing', m3.agg(F.sum('rows_inserted') + F.sum('rows_updated')).first()[0] == 0)
finally:
    for t in TMP.values():
        spark.sql(f'DROP TABLE IF EXISTS {t}')

display(spark.createDataFrame(inc_results, 'test_id string, description string, result string').orderBy('test_id'))
inc_failed = [r for r in inc_results if r[2] != 'PASS']
assert not inc_failed, f'incremental tests failed: {inc_failed}'
print('ALL', len(inc_results), 'INCREMENTAL TESTS PASSED')

## Step H - Delta history, time travel, Spark plan and skew

In [ ]:
# DELTA-ONLY: version history and time travel of the fact and the daily mart
for name in [TARGETS['fact_flight'], TARGETS['mart_daily']]:
    print('History of', name)
    display(spark.sql(f'DESCRIBE HISTORY {name}')
            .select('version', 'timestamp', 'operation',
                    F.col('operationMetrics')['numTargetRowsInserted'].alias('rows_inserted'),
                    F.col('operationMetrics')['numTargetRowsUpdated'].alias('rows_updated'),
                    F.col('operationMetrics')['numTargetRowsDeleted'].alias('rows_deleted'))
            .orderBy('version'))

fact_name = TARGETS['fact_flight']
hist_rows = spark.sql(f'DESCRIBE HISTORY {fact_name}').select('version', 'operation').collect()
v_first_load = min(r.version for r in hist_rows if r.operation == 'MERGE')     # first MERGE = first data load
v_latest = max(r.version for r in hist_rows)
biz_cols = [c for c in spark.table(fact_name).columns if c not in AUDIT_COLS]
old = spark.sql(f'SELECT * FROM {fact_name} VERSION AS OF {v_first_load}').select(*biz_cols)
new = spark.table(fact_name).select(*biz_cols)
print(f'Time travel: version {v_first_load} rows = {old.count()}, latest version {v_latest} rows = {new.count()}')
print('Rows that differ between the first load and now (replay must add none):', new.exceptAll(old).count() + old.exceptAll(new).count())
# Rollback (documentation only - do not run in the demo):  RESTORE TABLE <fact> TO VERSION AS OF <n>

In [ ]:
# Spark plan of a representative dashboard query (airport punctuality for one week)
REP_QUERY = f"""
SELECT a.airport_name, f.flight_date,
       SUM(CAST(f.is_on_time AS INT)) AS on_time, SUM(CAST(f.is_operated AS INT)) AS operated,
       ROUND(100.0 * SUM(CAST(f.is_on_time AS INT)) / SUM(CAST(f.is_operated AS INT)), 2) AS on_time_pct
FROM {TARGETS['fact_flight']} f JOIN {TARGETS['dim_airport']} a ON f.airport_sk = a.airport_sk
WHERE f.flight_date BETWEEN DATE'2026-07-10' AND DATE'2026-07-16'
GROUP BY a.airport_name, f.flight_date"""
plan = spark.sql('EXPLAIN FORMATTED ' + REP_QUERY).first()[0]
print(plan[:3000])
print('Broadcast join used for the dimension:', 'BroadcastHashJoin' in plan)

# Skew: how evenly flights spread over the join / group keys
for col_name in ['airport_sk', 'airline_sk', 'flight_date']:
    s = (spark.table(TARGETS['fact_flight']).groupBy(col_name).count()
         .agg(F.min('count').alias('min'), F.max('count').alias('max'), F.round(F.avg('count'), 1).alias('avg')).first())
    print(f'{col_name:12s} rows per key: min {s["min"]} | max {s["max"]} | avg {s["avg"]} | max/avg {s["max"] / s["avg"]:.2f}')

## Step I - Benchmark a representative query before / after OPTIMIZE (TC10)

In [ ]:
# DELTA-ONLY: timings are recorded as measured - no improvement is claimed unless the numbers show it.
def time_query(sql, runs=3):
    out = []
    for _ in range(runs):
        t0 = time.perf_counter()
        spark.sql(sql).collect()
        out.append(round(time.perf_counter() - t0, 3))
    return sorted(out)[len(out) // 2], out      # median, all runs

if RUN_BENCHMARK:
    try:
        spark.conf.set('spark.databricks.io.cache.enabled', 'false')   # not allowed on some compute types
    except Exception as e:
        print('disk cache setting not changed:', type(e).__name__)
    files_before = spark.sql(f"DESCRIBE DETAIL {TARGETS['fact_flight']}").first().numFiles
    med_before, runs_before = time_query(REP_QUERY)
    spark.sql(f"OPTIMIZE {TARGETS['fact_flight']} ZORDER BY (flight_date, airport_sk)")
    files_after = spark.sql(f"DESCRIBE DETAIL {TARGETS['fact_flight']}").first().numFiles
    med_after, runs_after = time_query(REP_QUERY)
    bench = spark.createDataFrame([(BATCH_ID, 'airport punctuality - one week', files_before, files_after, med_before, med_after,
                                    str(runs_before), str(runs_after))],
                                  'batch_id string, query string, files_before long, files_after long, median_sec_before double, '
                                  'median_sec_after double, runs_before string, runs_after string')
    bench = bench.withColumn('measured_at', F.current_timestamp())
    bench.write.format('delta').mode('append').saveAsTable(BENCH_TABLE)
    BENCH_RAN = True
    display(bench)
    verdict = 'faster' if med_after < med_before else 'not faster (data is small; startup cost dominates)'
    print(f'files {files_before} -> {files_after} | median {med_before}s -> {med_after}s | after OPTIMIZE the query was {verdict}')
else:
    print('Benchmark skipped (RUN_BENCHMARK = False)')

## Step J - Table comments for the dashboard and Genie, and KPI card queries (TC09)

In [ ]:
comments = {
    'dim_airline': 'Conformed airline dimension (SCD1). airline_sk = xxhash64(airline_id); -1 = UNKNOWN member.',
    'dim_airport': 'Conformed airport dimension (SCD1). airport_sk = xxhash64(airport_id); -1 = UNKNOWN member.',
    'dim_date': 'Calendar dimension. date_key = yyyyMMdd.',
    'dim_status': 'Flight status codes. is_cancelled drives the cancellation rate; REVERSED is not a cancellation.',
    'fact_flight': 'One row per flight_id with its latest valid status. departure_delay_minutes = Silver delay_minutes. '
                   f'is_on_time = not cancelled and delay <= {ON_TIME_THRESHOLD_MIN} min. Times are UTC.',
    'fact_transitions': 'One row per flight_id + source_sequence: sequence 0 = initial status, others = CDC changes.',
    'fact_secondary': 'One row per secondary operational event (ALERT, CHECK, REVIEW) linked to a flight.',
    'mart_daily': 'Daily network KPIs: total, cancelled, operated, on-time flights, cancellation rate %, on-time %, average departure delay.',
    'mart_airport': 'Daily airport punctuality: on_time_performance_pct = on-time / operated flights at the airport.',
    'mart_airline': 'Daily airline KPIs for delay attribution.',
    'run_log': 'Rows inserted / updated per Gold run and table (freshness monitoring).',
}
for key, text in comments.items():
    spark.sql(f"COMMENT ON TABLE {TARGETS[key]} IS '{text}'")
print('Table comments set on', len(comments), 'Gold tables.')

# KPI card queries - run these in the SQL editor / dashboard and compare with the values below (TC09)
KPI_CARDS = {
    'Total flights': f"SELECT SUM(total_flights) FROM {TARGETS['mart_daily']}",
    'Cancellation rate %': f"SELECT ROUND(100.0 * SUM(cancelled_flights) / SUM(total_flights), 2) FROM {TARGETS['mart_daily']}",
    'On-time performance %': f"SELECT ROUND(100.0 * SUM(on_time_flights) / SUM(operated_flights), 2) FROM {TARGETS['mart_daily']}",
    'Avg departure delay (min)': f"SELECT ROUND(SUM(total_departure_delay_minutes) / SUM(operated_flights), 2) FROM {TARGETS['mart_daily']}",
    'Least punctual airport': f"""SELECT airport_id FROM (
                                    SELECT airport_id, SUM(on_time_flights) / SUM(operated_flights) AS on_time_ratio
                                    FROM {TARGETS['mart_airport']} GROUP BY airport_id HAVING SUM(operated_flights) >= 50)
                                  ORDER BY on_time_ratio, airport_id LIMIT 1""",
}
display(spark.createDataFrame([(k, str(spark.sql(q).first()[0]), ' '.join(q.split())) for k, q in KPI_CARDS.items()],
                              'kpi string, value string, sql string'))

## Step K - Final validation (read-only)

In [ ]:
# ============================================================
# DAY 3 FINAL VALIDATION - read-only checks of the Gold layer
# ============================================================
vresults = []

def vcheck(name, condition, detail=''):
    vresults.append((name, 'PASS' if condition else 'FAIL', detail))
    print(('PASS' if condition else 'FAIL') + ' | ' + name + (f' | {detail}' if detail else ''))

def vinfo(name, detail):
    vresults.append((name, 'INFO', detail))
    print(f'INFO | {name} | {detail}')

for name in ALL_GOLD_TABLES:
    if name == BENCH_TABLE and not BENCH_RAN:
        continue
    vcheck('Table exists: ' + name.split('.')[-1], spark.catalog.tableExists(name))

fact = spark.table(TARGETS['fact_flight'])
n_fact = fact.count()
vcheck('fact_flight_activity unique at flight_id', n_fact == fact.select('flight_id').distinct().count(), f'{n_fact} rows')
vcheck('fact rows = silver_flight_current rows', n_fact == spark.table(SILVER_FLIGHT_CURRENT).count())
vcheck('No unmatched airline / airport keys', fact.filter((F.col('airline_sk') == UNKNOWN_SK) | (F.col('airport_sk') == UNKNOWN_SK)).count() == 0)
tr = spark.table(TARGETS['fact_transitions'])
vcheck('fact_status_transitions unique at flight_id + source_sequence', tr.count() == tr.select('flight_id', 'source_sequence').distinct().count(), f'{tr.count()} rows')
fs = spark.table(TARGETS['fact_secondary'])
vcheck('fact_secondary_events unique at event_id', fs.count() == fs.select('event_id').distinct().count(), f'{fs.count()} rows')
for key, grain in [('mart_daily', ['flight_date']), ('mart_airport', ['flight_date', 'airport_sk']), ('mart_airline', ['flight_date', 'airline_sk'])]:
    m = spark.table(TARGETS[key])
    vcheck(f'{TARGETS[key].split(".")[-1]} unique at {"+".join(grain)}', m.count() == m.select(*grain).distinct().count(), f'{m.count()} rows')
    vcheck(f'{TARGETS[key].split(".")[-1]} total flights = fact', m.agg(F.sum('total_flights')).first()[0] == n_fact)
vcheck('Reconciliation (this batch) all PASS', spark.table(RECON_TABLE).filter((F.col('batch_id') == BATCH_ID) & (F.col('result') != 'PASS')).count() == 0)
vcheck('Replay batch changed 0 rows', spark.table(TARGETS['run_log']).filter(F.col('batch_id') == REPLAY_BATCH)
       .agg(F.sum('rows_inserted') + F.sum('rows_updated')).first()[0] == 0)
vcheck('Incremental tests all passed', len(inc_results) > 0 and all(r[2] == 'PASS' for r in inc_results), f'{len(inc_results)} tests')
vcheck('KPI definitions saved', spark.table(KPI_DEFS_TABLE).count() == len(kpi_rows))
if STRICT_EXPECTED_COUNTS:
    vcheck('Capstone counts: fact 11,500 / transitions 12,020 / secondary 310', (n_fact, tr.count(), fs.count()) == (11500, 12020, 310))
    vcheck('Capstone counts: marts 32 / 4,800 / 3,520',
           tuple(spark.table(TARGETS[k]).count() for k in ['mart_daily', 'mart_airport', 'mart_airline']) == (32, 4800, 3520))
freshness = spark.table(TARGETS['run_log']).agg(F.max('run_ts')).first()[0]
vinfo('Freshness', f'last Gold run (UTC): {freshness}')
vinfo('On-time threshold', f'{ON_TIME_THRESHOLD_MIN} minutes - assumption, needs client approval')
vinfo('Departure delay', 'source has one delay field (delay_minutes); it is used as departure delay')
vinfo('Cancellation date', 'cancellations are counted on the flight date, not on the date the CDC change arrived')

passed = sum(1 for r in vresults if r[1] == 'PASS')
failed = sum(1 for r in vresults if r[1] == 'FAIL')
print('=' * 65)
print('DAY 3 GOLD VALIDATION SUMMARY | passed:', passed, '| failed:', failed, '| info:', sum(1 for r in vresults if r[1] == 'INFO'))
print('RESULT:', 'ALL EXECUTED VALIDATION CHECKS PASSED' if failed == 0 else 'VALIDATION NEEDS ATTENTION')
print('=' * 65)
for name, status, detail in vresults:
    if status == 'FAIL':
        print('-', name, detail)
assert failed == 0

# Day 3 - Gold layer: decisions and limitations

1. **Fact grain:** `fact_flight_activity` = one row per `flight_id` (current state from `silver_flight_current`); `fact_status_transitions` = one row per `flight_id + source_sequence` (sequence 0 = initial status); `fact_secondary_events` = one row per `event_id`. Each grain is asserted unique.
2. **Surrogate keys:** `airline_sk` / `airport_sk` = `xxhash64(business key)`. They are deterministic, so a rerun or a rebuild gives the same keys and facts never need re-keying. `-1` is the UNKNOWN member; any fact row on `-1` is an unmatched business key (TC07 expects 0).
3. **Dimensions are SCD1:** the reference extracts carry one version per key (`effective_date` 2026-01-01). If an attribute changes, the dimension row is updated and all marts are recomputed automatically.
4. **Departure delay:** the source has a single `delay_minutes` field; it is treated as departure delay. Cancelled flights are excluded from delay and punctuality (they did not operate).
5. **On-time threshold = 15 minutes** (industry A15 convention). It is a parameter (`ON_TIME_THRESHOLD_MIN`) and needs client approval.
6. **Cancellation:** only `CANCELLED` counts. `REVERSED` (173 flights) is kept and reported as `reversed_flights` but is not a cancellation, same as Silver.
7. **Date attribution:** KPIs are by **flight date** (event date, UTC). Every CDC change is dated 2026-08-10, so a later cancellation updates the KPI of the original flight date, not 2026-08-10.
8. **Incremental logic:** every table is loaded with a hash-guarded `MERGE` (insert new keys, update only changed rows). The marts recompute only the flight dates touched by the batch. Replaying the same input inserts and updates **0** rows (Step F); the simulated later revision changes exactly one fact row and one mart date (Step G).
9. **Reconciliation:** expected values come from independent SQL over Silver, not from the Gold functions. Results are stored per batch in `gold_reconciliation`.
10. **Benchmark:** with ~11.5k rows the query is dominated by start-up cost, so OPTIMIZE may not be faster. The measured numbers are recorded as they are.
11. **Runtime:** the mart MERGE uses `WHEN NOT MATCHED BY SOURCE`, which needs Databricks Runtime 12.2 LTS or newer (or serverless).
12. **Schema changes:** a new Silver column will not reach Gold until it is added to the build function; run once with `RESET_GOLD = True` (or enable schema auto-merge) after such a change.